In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os
import random
import json
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from tqdm.auto import tqdm

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("PyTorch:", torch.__version__)

Mounted at /content/drive
Device: cuda
PyTorch: 2.11.0+cu130


In [2]:
# ============================================================
# EXPERIMENT: ORIGINAL NORMAL + SYNTHETIC ANEMIA
# ============================================================

PROJECT_ROOT = (
    "/content/drive/MyDrive/palm_video_project"
)

FUSION_ROOT = (
     "/content/drive/MyDrive/Fall Course Project/Fusion"
)

CONDITIONED_ROOT = os.path.join(
    PROJECT_ROOT,
    "palm_conditioned_binary_anemia"
)

MASTER_SPLIT_PATH = os.path.join(
    FUSION_ROOT,
    "fusion_master_split.csv"
)

TRAINING_MANIFEST_PATH = os.path.join(
    CONDITIONED_ROOT,
    "palm_cnn_augmented_training_manifest.csv"
)

REAL_TEST_MANIFEST_PATH = os.path.join(
    CONDITIONED_ROOT,
    "palm_cnn_final_test_manifest.csv"
)

SYNTHETIC_TRAIN_METADATA_PATH = os.path.join(
    CONDITIONED_ROOT,
    "synthetic_anemia_sd15",
    "full",
    "synthetic_metadata.csv"
)

SYNTHETIC_VAL_METADATA_PATH = os.path.join(
    CONDITIONED_ROOT,
    "synthetic_eval_sd15",
    "validation",
    "synthetic_validation_metadata.csv"
)

SYNTHETIC_TEST_METADATA_PATH = os.path.join(
    CONDITIONED_ROOT,
    "synthetic_eval_sd15",
    "test",
    "synthetic_test_metadata.csv"
)

VAL_QUALITY_PATH = os.path.join(
    CONDITIONED_ROOT,
    "synthetic_validation_frame_quality.csv"
)

TEST_QUALITY_PATH = os.path.join(
    CONDITIONED_ROOT,
    "synthetic_test_frame_quality.csv"
)

EXPERIMENT_ROOT = os.path.join(
    CONDITIONED_ROOT,
    "experiment_original_normal_synthetic_anemia"
)

os.makedirs(EXPERIMENT_ROOT, exist_ok=True)

HEAD_CHECKPOINT_PATH = os.path.join(
    EXPERIMENT_ROOT,
    "palm_cnn_original_normal_synthetic_anemia_head_best.pth"
)

FINETUNE_CHECKPOINT_PATH = os.path.join(
    EXPERIMENT_ROOT,
    "palm_cnn_original_normal_synthetic_anemia_finetuned_best.pth"
)

print("Experiment output directory:")
print(EXPERIMENT_ROOT)

required_files = [
    MASTER_SPLIT_PATH,
    TRAINING_MANIFEST_PATH,
    REAL_TEST_MANIFEST_PATH,
    SYNTHETIC_TRAIN_METADATA_PATH,
    SYNTHETIC_VAL_METADATA_PATH,
    SYNTHETIC_TEST_METADATA_PATH,
    VAL_QUALITY_PATH,
    TEST_QUALITY_PATH
]

for path in required_files:
    print(
        "FOUND  " if os.path.exists(path) else "MISSING ",
        path
    )

assert all(os.path.exists(p) for p in required_files), (
    "One or more required files are missing. "
    "Check the printed paths before proceeding."
)

Experiment output directory:
/content/drive/MyDrive/palm_video_project/palm_conditioned_binary_anemia/experiment_original_normal_synthetic_anemia
FOUND   /content/drive/MyDrive/Fall Course Project/Fusion/fusion_master_split.csv
FOUND   /content/drive/MyDrive/palm_video_project/palm_conditioned_binary_anemia/palm_cnn_augmented_training_manifest.csv
FOUND   /content/drive/MyDrive/palm_video_project/palm_conditioned_binary_anemia/palm_cnn_final_test_manifest.csv
FOUND   /content/drive/MyDrive/palm_video_project/palm_conditioned_binary_anemia/synthetic_anemia_sd15/full/synthetic_metadata.csv
FOUND   /content/drive/MyDrive/palm_video_project/palm_conditioned_binary_anemia/synthetic_eval_sd15/validation/synthetic_validation_metadata.csv
FOUND   /content/drive/MyDrive/palm_video_project/palm_conditioned_binary_anemia/synthetic_eval_sd15/test/synthetic_test_metadata.csv
FOUND   /content/drive/MyDrive/palm_video_project/palm_conditioned_binary_anemia/synthetic_validation_frame_quality.csv
FOUND

In [3]:
master_df = pd.read_csv(MASTER_SPLIT_PATH)
training_manifest_df = pd.read_csv(TRAINING_MANIFEST_PATH)
real_test_manifest_df = pd.read_csv(REAL_TEST_MANIFEST_PATH)

synthetic_train_meta = pd.read_csv(
    SYNTHETIC_TRAIN_METADATA_PATH
)

synthetic_val_meta = pd.read_csv(
    SYNTHETIC_VAL_METADATA_PATH
)

synthetic_test_meta = pd.read_csv(
    SYNTHETIC_TEST_METADATA_PATH
)

val_quality_df = pd.read_csv(VAL_QUALITY_PATH)
test_quality_df = pd.read_csv(TEST_QUALITY_PATH)

print("Master split:", master_df.shape)
print("Training manifest:", training_manifest_df.shape)
print("Real test manifest:", real_test_manifest_df.shape)
print("Synthetic training metadata:", synthetic_train_meta.shape)
print("Synthetic validation metadata:", synthetic_val_meta.shape)
print("Synthetic test metadata:", synthetic_test_meta.shape)

print("\nTraining manifest columns:")
print(training_manifest_df.columns.tolist())

print("\nReal test manifest columns:")
print(real_test_manifest_df.columns.tolist())

print("\nSynthetic training columns:")
print(synthetic_train_meta.columns.tolist())

print("\nSynthetic test columns:")
print(synthetic_test_meta.columns.tolist())

print("\nMaster split distribution:")
print(master_df["Fusion_Split"].value_counts())

print("\nSeverity distribution:")
print(
    master_df.groupby(
        ["Fusion_Split", "Severity_Class"]
    ).size()
)

Master split: (823, 7)
Training manifest: (3258, 8)
Real test manifest: (495, 7)
Synthetic training metadata: (642, 12)
Synthetic validation metadata: (396, 11)
Synthetic test metadata: (495, 12)

Training manifest columns:
['ID', 'Severity_Class', 'Anemia_Binary', 'frame_number', 'frame_name', 'image_path', 'source_type', 'source_patient']

Real test manifest columns:
['ID', 'Severity_Class', 'Anemia_Binary', 'frame_number', 'frame_name', 'image_path', 'source_type']

Synthetic training columns:
['ID', 'Severity_Class', 'Anemia_Binary', 'frame_number', 'frame_name', 'source_frame', 'synthetic_frame', 'seed', 'model', 'strength', 'guidance_scale', 'steps']

Synthetic test columns:
['ID', 'Severity_Class', 'Anemia_Binary', 'frame_number', 'frame_name', 'source_frame', 'synthetic_frame', 'seed', 'model', 'strength', 'guidance_scale', 'steps']

Master split distribution:
Fusion_Split
development    658
test           165
Name: count, dtype: int64

Severity distribution:
Fusion_Split  Seve

In [4]:
def standardize_id(value):
    value = str(value).strip()

    # Avoid converting IDs to floating-point strings
    if value.endswith(".0"):
        value = value[:-2]

    return value


def find_path_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col

    raise KeyError(
        f"No supported path column found. "
        f"Available columns: {df.columns.tolist()}"
    )


def normalize_image_path(path, base_dir=None):
    path = str(path)

    if os.path.isabs(path):
        return os.path.normpath(path)

    if base_dir is not None:
        return os.path.normpath(
            os.path.join(base_dir, path)
        )

    return os.path.normpath(path)


def is_valid_image(path):
    if not os.path.exists(path):
        return False

    try:
        with Image.open(path) as im:
            arr = np.asarray(im.convert("RGB"))

        gray = arr.mean(axis=2)
        black_fraction = float(np.mean(gray <= 5))
        mean_intensity = float(gray.mean())

        # Exclude fully black / almost completely black outputs
        if mean_intensity == 0.0 or black_fraction >= 0.99:
            return False

        return True

    except Exception:
        return False


def add_binary_label(df):
    df = df.copy()

    df["ID"] = df["ID"].map(standardize_id)

    df["Anemia_Binary"] = (
        df["Severity_Class"]
        .isin(["Leve", "Moderado"])
        .astype(int)
    )

    return df

In [5]:
master_df["ID"] = master_df["ID"].map(standardize_id)

development_df = master_df[
    master_df["Fusion_Split"].str.lower() == "development"
].copy()

test_patients_df = master_df[
    master_df["Fusion_Split"].str.lower() == "test"
].copy()

development_ids = set(development_df["ID"])
test_ids = set(test_patients_df["ID"])

# The validation metadata was generated for the previous
# patient-level development validation split.
synthetic_val_meta["ID"] = (
    synthetic_val_meta["ID"].map(standardize_id)
)

validation_ids = (
    set(synthetic_val_meta["ID"]) & development_ids
)

training_ids = development_ids - validation_ids

print("Development patients:", len(development_ids))
print("Training patients:", len(training_ids))
print("Validation patients:", len(validation_ids))
print("Final test patients:", len(test_ids))

print("\nPatient overlap checks:")
print("Train ∩ Validation:", len(training_ids & validation_ids))
print("Train ∩ Test:", len(training_ids & test_ids))
print("Validation ∩ Test:", len(validation_ids & test_ids))

assert len(training_ids & validation_ids) == 0
assert len(training_ids & test_ids) == 0
assert len(validation_ids & test_ids) == 0

assert training_ids | validation_ids == development_ids

print("\n✓ Patient-level split verified")

Development patients: 658
Training patients: 526
Validation patients: 132
Final test patients: 165

Patient overlap checks:
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0

✓ Patient-level split verified


In [6]:
assert "source_type" in training_manifest_df.columns, (
    "Expected source_type in the training manifest."
)

original_path_col = find_path_column(
    training_manifest_df,
    ["image_path", "path", "frame_path"]
)

original_df = training_manifest_df.copy()

original_df["ID"] = original_df["ID"].map(standardize_id)

original_df = original_df[
    (original_df["source_type"] == "real_original") &
    (original_df["Severity_Class"] == "Normal")
].copy()

original_df["image_path"] = original_df[
    original_path_col
].apply(normalize_image_path)

original_df = add_binary_label(original_df)

# Keep only Normal class images
assert original_df["Anemia_Binary"].eq(0).all()

original_train_df = original_df[
    original_df["ID"].isin(training_ids)
].copy()

original_val_df = original_df[
    original_df["ID"].isin(validation_ids)
].copy()

print("Original Normal training frames:", len(original_train_df))
print("Original Normal validation frames:", len(original_val_df))

print("Training Normal patients:",
      original_train_df["ID"].nunique())

print("Validation Normal patients:",
      original_val_df["ID"].nunique())

print("Missing original training files:",
      (~original_train_df["image_path"].apply(os.path.exists)).sum())

print("Missing original validation files:",
      (~original_val_df["image_path"].apply(os.path.exists)).sum())

assert original_train_df["image_path"].apply(os.path.exists).all()
assert original_val_df["image_path"].apply(os.path.exists).all()

Original Normal training frames: 1065
Original Normal validation frames: 267
Training Normal patients: 355
Validation Normal patients: 89
Missing original training files: 0
Missing original validation files: 0


In [7]:
synthetic_train_path_col = find_path_column(
    synthetic_train_meta,
    ["synthetic_frame", "image_path", "path"]
)

synthetic_train_df = synthetic_train_meta.copy()

synthetic_train_df["ID"] = (
    synthetic_train_df["ID"].map(standardize_id)
)

synthetic_train_df["image_path"] = synthetic_train_df[
    synthetic_train_path_col
].apply(
    lambda p: normalize_image_path(
        p,
        base_dir=os.path.dirname(SYNTHETIC_TRAIN_METADATA_PATH)
    )
)

synthetic_train_df = add_binary_label(synthetic_train_df)

# Only synthetic Anemia images
synthetic_train_df = synthetic_train_df[
    synthetic_train_df["Anemia_Binary"] == 1
].copy()

synthetic_train_df = synthetic_train_df[
    synthetic_train_df["ID"].isin(training_ids)
].copy()

# Verify paths and remove black/failed synthetic generations
synthetic_train_df["file_exists"] = (
    synthetic_train_df["image_path"].apply(os.path.exists)
)

synthetic_train_df["image_valid"] = (
    synthetic_train_df["image_path"].apply(is_valid_image)
)

print("Synthetic Anemia rows before filtering:",
      len(synthetic_train_df))

print("Missing synthetic files:",
      (~synthetic_train_df["file_exists"]).sum())

print("Invalid synthetic images:",
      (~synthetic_train_df["image_valid"]).sum())

synthetic_train_df = synthetic_train_df[
    synthetic_train_df["image_valid"]
].copy()

synthetic_train_df["source_type"] = "synthetic_anemia"

print("\nFinal synthetic Anemia training frames:",
      len(synthetic_train_df))

print("\nBy severity:")
print(synthetic_train_df["Severity_Class"].value_counts())

assert synthetic_train_df["Anemia_Binary"].eq(1).all()
assert synthetic_train_df["Severity_Class"].isin(
    ["Leve", "Moderado"]
).all()

Synthetic Anemia rows before filtering: 513
Missing synthetic files: 0
Invalid synthetic images: 0

Final synthetic Anemia training frames: 513

By severity:
Severity_Class
Leve        384
Moderado    129
Name: count, dtype: int64


In [8]:
val_quality_df["ID"] = val_quality_df["ID"].map(
    standardize_id
)

valid_val_paths = set(
    val_quality_df.loc[
        val_quality_df["valid"].astype(bool),
        "path"
    ].apply(os.path.normpath)
)

synthetic_val_path_col = find_path_column(
    synthetic_val_meta,
    ["synthetic_frame", "image_path", "path"]
)

synthetic_val_df = synthetic_val_meta.copy()

synthetic_val_df["ID"] = (
    synthetic_val_df["ID"].map(standardize_id)
)

synthetic_val_df["image_path"] = synthetic_val_df[
    synthetic_val_path_col
].apply(
    lambda p: normalize_image_path(
        p,
        base_dir=os.path.dirname(SYNTHETIC_VAL_METADATA_PATH)
    )
)

synthetic_val_df = add_binary_label(synthetic_val_df)

synthetic_val_df = synthetic_val_df[
    (synthetic_val_df["Anemia_Binary"] == 1) &
    (synthetic_val_df["ID"].isin(validation_ids))
].copy()

synthetic_val_df["quality_valid"] = (
    synthetic_val_df["image_path"].apply(os.path.normpath)
    .isin(valid_val_paths)
)

synthetic_val_df["file_exists"] = (
    synthetic_val_df["image_path"].apply(os.path.exists)
)

synthetic_val_df = synthetic_val_df[
    synthetic_val_df["quality_valid"] &
    synthetic_val_df["file_exists"]
].copy()

synthetic_val_df["source_type"] = "synthetic_anemia"

print("Synthetic Anemia validation frames:",
      len(synthetic_val_df))

print("Validation Anemia patients:",
      synthetic_val_df["ID"].nunique())

print("\nBy severity:")
print(synthetic_val_df["Severity_Class"].value_counts())

assert synthetic_val_df["Anemia_Binary"].eq(1).all()

Synthetic Anemia validation frames: 114
Validation Anemia patients: 43

By severity:
Severity_Class
Leve        78
Moderado    36
Name: count, dtype: int64


In [9]:
# ------------------------------------------------------------
# A. Original Normal test images
# ------------------------------------------------------------

real_test_path_col = find_path_column(
    real_test_manifest_df,
    ["image_path", "path", "frame_path"]
)

real_test_df = real_test_manifest_df.copy()

real_test_df["ID"] = real_test_df["ID"].map(
    standardize_id
)

real_test_df = add_binary_label(real_test_df)

real_test_df = real_test_df[
    (real_test_df["Severity_Class"] == "Normal") &
    (real_test_df["ID"].isin(test_ids))
].copy()

real_test_df["image_path"] = real_test_df[
    real_test_path_col
].apply(normalize_image_path)

real_test_df["source_type"] = "real_original_normal"

# ------------------------------------------------------------
# B. Valid synthetic Anemia test images
# ------------------------------------------------------------

test_quality_df["ID"] = test_quality_df["ID"].map(
    standardize_id
)

valid_test_paths = set(
    test_quality_df.loc[
        test_quality_df["valid"].astype(bool),
        "path"
    ].apply(os.path.normpath)
)

synthetic_test_path_col = find_path_column(
    synthetic_test_meta,
    ["synthetic_frame", "image_path", "path"]
)

synthetic_test_df = synthetic_test_meta.copy()

synthetic_test_df["ID"] = (
    synthetic_test_df["ID"].map(standardize_id)
)

synthetic_test_df["image_path"] = synthetic_test_df[
    synthetic_test_path_col
].apply(
    lambda p: normalize_image_path(
        p,
        base_dir=os.path.dirname(SYNTHETIC_TEST_METADATA_PATH)
    )
)

synthetic_test_df = add_binary_label(synthetic_test_df)

synthetic_test_df = synthetic_test_df[
    (synthetic_test_df["Anemia_Binary"] == 1) &
    (synthetic_test_df["ID"].isin(test_ids))
].copy()

synthetic_test_df["quality_valid"] = (
    synthetic_test_df["image_path"].apply(os.path.normpath)
    .isin(valid_test_paths)
)

synthetic_test_df["file_exists"] = (
    synthetic_test_df["image_path"].apply(os.path.exists)
)

synthetic_test_df = synthetic_test_df[
    synthetic_test_df["quality_valid"] &
    synthetic_test_df["file_exists"]
].copy()

synthetic_test_df["source_type"] = "synthetic_anemia"

# ------------------------------------------------------------
# C. Combine test sources
# ------------------------------------------------------------

test_manifest = pd.concat(
    [real_test_df, synthetic_test_df],
    ignore_index=True
)

# Keep the common columns
common_columns = [
    "ID",
    "Severity_Class",
    "Anemia_Binary",
    "image_path",
    "source_type"
]

test_manifest = test_manifest[common_columns].copy()

print("=" * 70)
print("NEW EXPERIMENT — TEST MANIFEST")
print("=" * 70)

print("Total frames:", len(test_manifest))
print("Unique patients:", test_manifest["ID"].nunique())

print("\nBy source and class:")
print(
    test_manifest.groupby(
        ["source_type", "Severity_Class"]
    ).size()
)

print("\nPatients per class:")
print(
    test_manifest.groupby("Anemia_Binary")["ID"].nunique()
)

print("\nMissing files:",
      (~test_manifest["image_path"].apply(os.path.exists)).sum())

assert test_manifest["image_path"].apply(os.path.exists).all()
assert test_manifest["Anemia_Binary"].isin([0, 1]).all()

assert (
    test_manifest.loc[
        test_manifest["source_type"] == "real_original_normal",
        "Anemia_Binary"
    ] == 0
).all()

assert (
    test_manifest.loc[
        test_manifest["source_type"] == "synthetic_anemia",
        "Anemia_Binary"
    ] == 1
).all()

test_manifest.to_csv(
    os.path.join(EXPERIMENT_ROOT, "test_manifest.csv"),
    index=False
)

print("\n✓ Test manifest saved")

NEW EXPERIMENT — TEST MANIFEST
Total frames: 484
Unique patients: 164

By source and class:
source_type           Severity_Class
real_original_normal  Normal            336
synthetic_anemia      Leve              109
                      Moderado           39
dtype: int64

Patients per class:
Anemia_Binary
0    112
1     52
Name: ID, dtype: int64

Missing files: 0

✓ Test manifest saved


In [10]:
# ============================================================
# CELL 10 — COMBINE TRAINING AND VALIDATION MANIFESTS
# ============================================================

# ------------------------------------------------------------
# TRAINING MANIFEST
# ------------------------------------------------------------

original_train_part = original_train_df[
    ["ID", "Severity_Class", "Anemia_Binary", "image_path"]
].copy()

original_train_part["source_type"] = "real_original_normal"

synthetic_train_part = synthetic_train_df[
    ["ID", "Severity_Class", "Anemia_Binary", "image_path"]
].copy()

synthetic_train_part["source_type"] = "synthetic_anemia"

train_manifest = pd.concat(
    [original_train_part, synthetic_train_part],
    ignore_index=True
)

# ------------------------------------------------------------
# VALIDATION MANIFEST
# ------------------------------------------------------------

original_val_part = original_val_df[
    ["ID", "Severity_Class", "Anemia_Binary", "image_path"]
].copy()

original_val_part["source_type"] = "real_original_normal"

synthetic_val_part = synthetic_val_df[
    ["ID", "Severity_Class", "Anemia_Binary", "image_path"]
].copy()

synthetic_val_part["source_type"] = "synthetic_anemia"

val_manifest = pd.concat(
    [original_val_part, synthetic_val_part],
    ignore_index=True
)

# ------------------------------------------------------------
# VERIFY FILES AND CLASS/SOURCE COMPOSITION
# ------------------------------------------------------------

assert train_manifest["image_path"].apply(os.path.exists).all()
assert val_manifest["image_path"].apply(os.path.exists).all()

# Training: Normal must be original; Anemia must be synthetic
assert (
    train_manifest.loc[
        train_manifest["Anemia_Binary"] == 0,
        "source_type"
    ] == "real_original_normal"
).all()

assert (
    train_manifest.loc[
        train_manifest["Anemia_Binary"] == 1,
        "source_type"
    ] == "synthetic_anemia"
).all()

# Validation: same source policy
assert (
    val_manifest.loc[
        val_manifest["Anemia_Binary"] == 0,
        "source_type"
    ] == "real_original_normal"
).all()

assert (
    val_manifest.loc[
        val_manifest["Anemia_Binary"] == 1,
        "source_type"
    ] == "synthetic_anemia"
).all()

# Verify patient separation
assert set(train_manifest["ID"]).isdisjoint(
    set(val_manifest["ID"])
)

assert set(train_manifest["ID"]).isdisjoint(test_ids)
assert set(val_manifest["ID"]).isdisjoint(test_ids)

# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print("=" * 70)
print("TRAINING MANIFEST")
print("=" * 70)

print("Frames:", len(train_manifest))
print("Patients:", train_manifest["ID"].nunique())

print("\nSource counts:")
print(train_manifest["source_type"].value_counts())

print("\nClass counts:")
print(train_manifest["Anemia_Binary"].value_counts().sort_index())

print("\nSource by class:")
print(pd.crosstab(
    train_manifest["source_type"],
    train_manifest["Anemia_Binary"]
))

print("\n" + "=" * 70)
print("VALIDATION MANIFEST")
print("=" * 70)

print("Frames:", len(val_manifest))
print("Patients:", val_manifest["ID"].nunique())

print("\nSource counts:")
print(val_manifest["source_type"].value_counts())

print("\nClass counts:")
print(val_manifest["Anemia_Binary"].value_counts().sort_index())

print("\nSource by class:")
print(pd.crosstab(
    val_manifest["source_type"],
    val_manifest["Anemia_Binary"]
))

# Save corrected manifests
train_manifest.to_csv(
    os.path.join(EXPERIMENT_ROOT, "train_manifest.csv"),
    index=False
)

val_manifest.to_csv(
    os.path.join(EXPERIMENT_ROOT, "validation_manifest.csv"),
    index=False
)

print("\n✓ All source/class assertions passed")
print("✓ No patient overlap between training, validation, and test")
print("✓ Corrected manifests saved")

TRAINING MANIFEST
Frames: 1578
Patients: 526

Source counts:
source_type
real_original_normal    1065
synthetic_anemia         513
Name: count, dtype: int64

Class counts:
Anemia_Binary
0    1065
1     513
Name: count, dtype: int64

Source by class:
Anemia_Binary            0    1
source_type                    
real_original_normal  1065    0
synthetic_anemia         0  513

VALIDATION MANIFEST
Frames: 381
Patients: 132

Source counts:
source_type
real_original_normal    267
synthetic_anemia        114
Name: count, dtype: int64

Class counts:
Anemia_Binary
0    267
1    114
Name: count, dtype: int64

Source by class:
Anemia_Binary           0    1
source_type                   
real_original_normal  267    0
synthetic_anemia        0  114

✓ All source/class assertions passed
✓ No patient overlap between training, validation, and test
✓ Corrected manifests saved


In [11]:
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10,
        hue=0.02
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [12]:
class PalmFrameDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        with Image.open(row["image_path"]) as im:
            image = im.convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        label = torch.tensor(
            float(row["Anemia_Binary"]),
            dtype=torch.float32
        )

        return image, label


train_dataset = PalmFrameDataset(
    train_manifest,
    transform=train_transform
)

val_dataset = PalmFrameDataset(
    val_manifest,
    transform=eval_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Training batches: 50
Validation batches: 12


In [13]:
weights = models.EfficientNet_B0_Weights.DEFAULT

model = models.efficientnet_b0(weights=weights)

in_features = model.classifier[1].in_features

model.classifier[1] = nn.Linear(in_features, 1)

model = model.to(device)

# Compute class weighting from training frames only
negative_count = int(
    (train_manifest["Anemia_Binary"] == 0).sum()
)

positive_count = int(
    (train_manifest["Anemia_Binary"] == 1).sum()
)

assert positive_count > 0 and negative_count > 0

pos_weight = torch.tensor(
    [negative_count / positive_count],
    dtype=torch.float32,
    device=device
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

print("Model: EfficientNet-B0")
print("Device:", device)
print("Training Normal frames:", negative_count)
print("Training Anemia frames:", positive_count)
print("Positive class weight:", pos_weight.item())

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 124MB/s]


Model: EfficientNet-B0
Device: cuda
Training Normal frames: 1065
Training Anemia frames: 513
Positive class weight: 2.0760233402252197


In [14]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()

    total_loss = 0.0
    total_samples = 0

    for images, labels in tqdm(loader, leave=False):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(images).squeeze(1)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / max(total_samples, 1)


@torch.no_grad()
def validate_one_epoch(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_labels = []
    all_probs = []

    for images, labels in tqdm(loader, leave=False):
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images).squeeze(1)
        loss = criterion(logits, labels)

        probs = torch.sigmoid(logits)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    return (
        total_loss / max(total_samples, 1),
        np.asarray(all_labels),
        np.asarray(all_probs)
    )


def run_training(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    epochs,
    checkpoint_path,
    training_type
):
    history = []
    best_val_loss = float("inf")

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(
            model, train_loader, criterion, optimizer
        )

        val_loss, y_val, p_val = validate_one_epoch(
            model, val_loader, criterion
        )

        val_pred = (p_val >= 0.5).astype(int)

        row = {
            "epoch": epoch,
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
            "val_accuracy": float(
                accuracy_score(y_val, val_pred)
            ),
            "val_recall": float(
                recall_score(y_val, val_pred, zero_division=0)
            ),
            "val_f1": float(
                f1_score(y_val, val_pred, zero_division=0)
            )
        }

        try:
            row["val_roc_auc"] = float(
                roc_auc_score(y_val, p_val)
            )
        except ValueError:
            row["val_roc_auc"] = None

        history.append(row)

        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_acc={row['val_accuracy']:.4f} | "
            f"val_recall={row['val_recall']:.4f} | "
            f"val_f1={row['val_f1']:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_loss": best_val_loss,
                    "training_type": training_type
                },
                checkpoint_path
            )

            print("  ✓ Best checkpoint saved")

    return pd.DataFrame(history)

In [15]:
for param in model.features.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

optimizer = torch.optim.Adam(
    model.classifier.parameters(),
    lr=1e-3
)

head_history = run_training(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    epochs=10,
    checkpoint_path=HEAD_CHECKPOINT_PATH,
    training_type="head_only"
)

head_history.to_csv(
    os.path.join(EXPERIMENT_ROOT, "head_training_history.csv"),
    index=False
)

print("\nHead training completed.")
print("Checkpoint:", HEAD_CHECKPOINT_PATH)

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

Epoch 01/10 | train_loss=0.5206 | val_loss=0.2448 | val_acc=0.9869 | val_recall=0.9912 | val_f1=0.9784
  ✓ Best checkpoint saved


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

...

  0%|          | 0/12 [00:00<?, ?it/s]

Epoch 09/10 | train_loss=0.0645 | val_loss=0.0364 | val_acc=0.9948 | val_recall=0.9912 | val_f1=0.9912
  ✓ Best checkpoint saved


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

Epoch 10/10 | train_loss=0.0559 | val_loss=0.0311 | val_acc=0.9974 | val_recall=0.9912 | val_f1=0.9956
  ✓ Best checkpoint saved

Head training completed.
Checkpoint: /content/drive/MyDrive/palm_video_project/palm_conditioned_binary_anemia/experiment_original_normal_synthetic_anemia/palm_cnn_original_normal_synthetic_anemia_head_best.pth


In [16]:
head_checkpoint = torch.load(
    HEAD_CHECKPOINT_PATH,
    map_location=device
)

model.load_state_dict(
    head_checkpoint["model_state_dict"]
)

# Unfreeze all layers
for param in model.parameters():
    param.requires_grad = True

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

finetune_history = run_training(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    epochs=10,
    checkpoint_path=FINETUNE_CHECKPOINT_PATH,
    training_type="full_finetune"
)

finetune_history.to_csv(
    os.path.join(EXPERIMENT_ROOT, "finetune_training_history.csv"),
    index=False
)

print("\nFine-tuning completed.")
print("Best checkpoint:", FINETUNE_CHECKPOINT_PATH)

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

Epoch 01/10 | train_loss=0.0203 | val_loss=0.0013 | val_acc=1.0000 | val_recall=1.0000 | val_f1=1.0000
  ✓ Best checkpoint saved


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

...

  0%|          | 0/12 [00:00<?, ?it/s]

Epoch 09/10 | train_loss=0.0003 | val_loss=0.0000 | val_acc=1.0000 | val_recall=1.0000 | val_f1=1.0000
  ✓ Best checkpoint saved


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

Epoch 10/10 | train_loss=0.0017 | val_loss=0.0000 | val_acc=1.0000 | val_recall=1.0000 | val_f1=1.0000
  ✓ Best checkpoint saved

Fine-tuning completed.
Best checkpoint: /content/drive/MyDrive/palm_video_project/palm_conditioned_binary_anemia/experiment_original_normal_synthetic_anemia/palm_cnn_original_normal_synthetic_anemia_finetuned_best.pth


In [17]:
checkpoint = torch.load(
    FINETUNE_CHECKPOINT_PATH,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(device)
model.eval()

print("Loaded checkpoint:", FINETUNE_CHECKPOINT_PATH)
print("Best epoch:", checkpoint["epoch"])
print("Validation loss:", checkpoint["val_loss"])

Loaded checkpoint: /content/drive/MyDrive/palm_video_project/palm_conditioned_binary_anemia/experiment_original_normal_synthetic_anemia/palm_cnn_original_normal_synthetic_anemia_finetuned_best.pth
Best epoch: 10
Validation loss: 2.7113601962454526e-05


In [18]:
test_dataset = PalmFrameDataset(
    test_manifest,
    transform=eval_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

@torch.no_grad()
def predict_test_frames(model, loader, dataframe):
    model.eval()

    probabilities = []

    for images, _ in tqdm(loader):
        images = images.to(device)

        logits = model(images).squeeze(1)
        probs = torch.sigmoid(logits)

        probabilities.extend(
            probs.cpu().numpy().tolist()
        )

    assert len(probabilities) == len(dataframe)

    predictions_df = dataframe.reset_index(
        drop=True
    ).copy()

    predictions_df["Anemia_Probability"] = probabilities

    return predictions_df


frame_predictions_df = predict_test_frames(
    model,
    test_loader,
    test_manifest
)

print("Test frame predictions:", len(frame_predictions_df))

display(frame_predictions_df.head())

frame_predictions_df.to_csv(
    os.path.join(
        EXPERIMENT_ROOT,
        "test_frame_predictions.csv"
    ),
    index=False
)

  0%|          | 0/16 [00:00<?, ?it/s]

Test frame predictions: 484


,ID,Severity_Class,Anemia_Binary,image_path,source_type,Anemia_Probability
0,ID007,Normal,0,/content/drive/MyDrive/palm_video_project/vide...,real_original_normal,4.285519e-06
1,ID007,Normal,0,/content/drive/MyDrive/palm_video_project/vide...,real_original_normal,1.923059e-06
2,ID007,Normal,0,/content/drive/MyDrive/palm_video_project/vide...,real_original_normal,5.528297e-06
3,ID009,Normal,0,/content/drive/MyDrive/palm_video_project/vide...,real_original_normal,1.974401e-08
4,ID009,Normal,0,/content/drive/MyDrive/palm_video_project/vide...,real_original_normal,1.331655e-08


In [19]:
patient_predictions_df = (
    frame_predictions_df
    .groupby(
        ["ID", "Severity_Class", "Anemia_Binary"],
        as_index=False
    )
    .agg(
        Anemia_Probability=("Anemia_Probability", "mean"),
        Frames_Used=("Anemia_Probability", "count"),
        Source_Types=("source_type",
                      lambda x: ", ".join(sorted(set(x))))
    )
)

patient_predictions_df["Predicted_Binary"] = (
    patient_predictions_df["Anemia_Probability"] >= 0.5
).astype(int)

print("Evaluated patients:", len(patient_predictions_df))

print("\nPatients by actual class:")
print(patient_predictions_df["Anemia_Binary"].value_counts())

print("\nAvailable frames per patient:")
print(patient_predictions_df["Frames_Used"].value_counts().sort_index())

display(patient_predictions_df.head())

patient_predictions_df.to_csv(
    os.path.join(
        EXPERIMENT_ROOT,
        "test_patient_predictions.csv"
    ),
    index=False
)

Evaluated patients: 164

Patients by actual class:
Anemia_Binary
0    112
1     52
Name: count, dtype: int64

Available frames per patient:
Frames_Used
1      2
2      4
3    158
Name: count, dtype: int64


,ID,Severity_Class,Anemia_Binary,Anemia_Probability,Frames_Used,Source_Types,Predicted_Binary
0,ID005,Moderado,1,9.999985e-01,2,synthetic_anemia,1
1,ID007,Normal,0,3.912292e-06,3,real_original_normal,0
2,ID009,Normal,0,1.371796e-08,3,real_original_normal,0
3,ID010,Moderado,1,9.999999e-01,3,synthetic_anemia,1
4,ID013,Normal,0,3.791728e-07,3,real_original_normal,0


In [20]:
y_true = patient_predictions_df["Anemia_Binary"].to_numpy()
y_prob = patient_predictions_df["Anemia_Probability"].to_numpy()
y_pred = patient_predictions_df["Predicted_Binary"].to_numpy()

metrics = {
    "experiment": "original_normal_synthetic_anemia",
    "patients_evaluated": int(len(patient_predictions_df)),
    "test_frames": int(len(frame_predictions_df)),
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "precision": float(
        precision_score(y_true, y_pred, zero_division=0)
    ),
    "recall": float(
        recall_score(y_true, y_pred, zero_division=0)
    ),
    "f1_score": float(
        f1_score(y_true, y_pred, zero_division=0)
    ),
    "confusion_matrix": confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).tolist(),
    "threshold": 0.5,
    "evaluation_policy": (
        "Original Normal test frames plus valid synthetic "
        "Leve and Moderado test frames. Patient-level mean "
        "probability; failed synthetic images excluded."
    )
}

try:
    metrics["roc_auc"] = float(
        roc_auc_score(y_true, y_prob)
    )
except ValueError:
    metrics["roc_auc"] = None

print("=" * 70)
print("NEW EXPERIMENT — FINAL PATIENT-LEVEL TEST RESULTS")
print("=" * 70)

print("Patients evaluated:", metrics["patients_evaluated"])
print("Test frames:", metrics["test_frames"])
print(f"Accuracy : {metrics['accuracy']:.4f}")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall   : {metrics['recall']:.4f}")
print(f"F1 Score : {metrics['f1_score']:.4f}")
print(f"ROC-AUC  : {metrics['roc_auc']}")

print("\nConfusion matrix:")
print(np.array(metrics["confusion_matrix"]))

print("\nClassification report:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["Not Anemia", "Anemia"],
        zero_division=0
    )
)

metrics_path = os.path.join(
    EXPERIMENT_ROOT,
    "test_metrics.json"
)

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("\nMetrics saved to:")
print(metrics_path)

NEW EXPERIMENT — FINAL PATIENT-LEVEL TEST RESULTS
Patients evaluated: 164
Test frames: 484
Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1 Score : 1.0000
ROC-AUC  : 1.0

Confusion matrix:
[[112   0]
 [  0  52]]

Classification report:
              precision    recall  f1-score   support

  Not Anemia       1.00      1.00      1.00       112
      Anemia       1.00      1.00      1.00        52

    accuracy                           1.00       164
   macro avg       1.00      1.00      1.00       164
weighted avg       1.00      1.00      1.00       164


Metrics saved to:
/content/drive/MyDrive/palm_video_project/palm_conditioned_binary_anemia/experiment_original_normal_synthetic_anemia/test_metrics.json
